In [3]:
class LiquidityPreferenceEconomy:
    def __init__(self, YD_0, V_0, B_h_0, BL_h_0, pBL_0, r_b_0=0.0):
        #H_0 is residual: whatever wealth isn't held as bills or bonds starts out as money.
        H_0 = V_0 - B_h_0 - pBL_0 * BL_h_0
        #V_0 must be enough to cover the initial bill and bond holdings.
        if H_0 < -1e-9:
            raise ValueError("V_0 must cover B_h_0 plus pBL_0 * BL_h_0.")

        self.period = [0]
        self.G = [0.0]
        self.Y = [0.0]
        self.T = [0.0]
        self.YD = [float(YD_0)]
        self.YD_e = [float(YD_0)]
        self.C = [0.0]
        self.CG = [0.0]
        self.V = [float(V_0)]
        self.V_e = [float(V_0)]
        self.H_d = [float(H_0)]
        self.B_d = [float(B_h_0)]
        self.BL_d = [float(BL_h_0)]
        self.H_h = [float(H_0)]
        self.B_h = [float(B_h_0)]
        self.BL_h = [float(BL_h_0)]
        self.H_s = [float(H_0)]
        self.B_cb = [float(H_0)]
        self.B_s = [float(B_h_0 + H_0)]
        self.BL_s = [float(BL_h_0)]
        self.delta_Bs = [0.0]
        self.delta_Hs = [0.0]
        self.r_b = [float(r_b_0)]
        self.pBL = [float(pBL_0)]
        self.ER_BL = [0.0]

    @staticmethod
    #Equations 5.9A, 5.10 and 5.11: portfolio shares of expected wealth.
    def _shares(lambda_h, lambda_b, lambda_bl, r_b, expected_return, ratio):
        h_share = lambda_h[0] + lambda_h[1] * r_b + lambda_h[2] * expected_return + lambda_h[3] * ratio
        b_share = lambda_b[0] + lambda_b[1] * r_b + lambda_b[2] * expected_return + lambda_b[3] * ratio
        bl_share = lambda_bl[0] + lambda_bl[1] * r_b + lambda_bl[2] * expected_return + lambda_bl[3] * ratio
        if min(h_share, b_share, bl_share) < -1e-9 or abs(h_share + b_share + bl_share - 1) > 1e-6:
            raise ValueError(
                "Portfolio shares must be non-negative and sum to one. "
                f"They are H={h_share:.4f}, B={b_share:.4f}, BL={bl_share:.4f}."
            )
        return h_share, b_share, bl_share

    #lambda_h, lambda_b and lambda_bl are four-number tuples: (constant,
    #coefficient on r_b, coefficient on ER_BL, coefficient on YD_e/V_e).
    #Signed coefficients are allowed.
    def run_period(self, G, alpha1, alpha2, theta, r_b, pBL,
                   lambda_h, lambda_b, lambda_bl):
        if pBL <= 0:
            raise ValueError("pBL must be positive.")
        if any(len(x) != 4 for x in (lambda_h, lambda_b, lambda_bl)):
            raise ValueError("Each portfolio coefficient vector must contain four values.")

        YD_e = self.YD[-1]                       #Equation 5.22
        V_1, B_h_1, BL_h_1 = self.V[-1], self.B_h[-1], self.BL_h[-1]
        B_s_1, B_cb_1 = self.B_s[-1], self.B_cb[-1]
        r_b_1, pBL_1 = self.r_b[-1], self.pBL[-1]

        #Equations 5.4-5.7 (the expected capital gain is zero in base LP).
        CG = (pBL - pBL_1) * BL_h_1
        C = alpha1 * YD_e + alpha2 * V_1
        Y = C + G
        household_interest = r_b_1 * B_h_1 + BL_h_1
        T = theta * (Y + household_interest)
        YD = Y - T + household_interest
        V = V_1 + (YD - C) + CG
        V_e = V_1 + (YD_e - C) + CG

        #Equation 5.19: expected return on the long bond (coupon = 1).
        expected_return = 1 / pBL_1
        if V_e == 0:
            raise ZeroDivisionError("Expected wealth V_e cannot be zero.")
        income_wealth_ratio = YD_e / V_e
        h_share, b_share, bl_share = self._shares(
            lambda_h, lambda_b, lambda_bl, r_b_1, expected_return, income_wealth_ratio
        )

        #Equations 5.8-5.13. Long-bond demand is a quantity, not a value.
        H_d = h_share * V_e
        B_d = b_share * V_e
        BL_d = (bl_share * V_e) / pBL
        B_h, BL_h = B_d, BL_d                 #asset-market clearing

        #Equations 5.14-5.17: government and central-bank balance sheets.
        delta_Bs = (G + r_b_1 * B_s_1 + BL_h_1) - (T + r_b_1 * B_cb_1)
        B_s = B_s_1 + delta_Bs
        BL_s = BL_h
        B_cb = B_s - B_h
        delta_Hs = B_cb - B_cb_1
        H_s = self.H_s[-1] + delta_Hs
        H_h = H_s                              #hidden equation: H_h = H_s

        self.period.append(self.period[-1] + 1)
        values = {
            "G": G, "Y": Y, "T": T, "YD": YD, "YD_e": YD_e, "C": C,
            "CG": CG, "V": V, "V_e": V_e, "H_d": H_d, "B_d": B_d,
            "BL_d": BL_d, "H_h": H_h, "B_h": B_h, "BL_h": BL_h,
            "H_s": H_s, "B_cb": B_cb, "B_s": B_s, "BL_s": BL_s,
            "delta_Bs": delta_Bs, "delta_Hs": delta_Hs, "r_b": r_b,
            "pBL": pBL, "ER_BL": expected_return,
        }
        for name, value in values.items():
            getattr(self, name).append(float(value))

    def _print_table(self, indices, title, max_width=76):
        #Same plain style as Chapter 3; splits into blocks if periods overflow max_width.
        variables = [
            ("G", self.G), ("Y", self.Y), ("T", self.T), ("YD", self.YD),
            ("YD^e", self.YD_e), ("C", self.C), ("CG", self.CG), ("V", self.V),
            ("V^e", self.V_e), ("H^d", self.H_d), ("B^d", self.B_d),
            ("BL^d", self.BL_d), ("H_h", self.H_h), ("B_h", self.B_h),
            ("BL_h", self.BL_h), ("H_s", self.H_s), ("B_cb", self.B_cb),
            ("B_s", self.B_s), ("BL_s", self.BL_s), ("dB_s", self.delta_Bs),
            ("dH_s", self.delta_Hs), ("r_b", self.r_b), ("pBL", self.pBL),
            ("ER_BL", self.ER_BL),
        ]

        label_width, col_width = 10, 12

        indices = list(indices)
        cols_per_block = max(1, (max_width - label_width) // col_width)
        blocks = [indices[i:i + cols_per_block]
                  for i in range(0, len(indices), cols_per_block)]

        for block in blocks:
            header_line = f"{'Variable':<{label_width}}" + "".join(
                f"{'Period ' + str(self.period[i]):>{col_width}}" for i in block)
            total_width = len(header_line)

            block_title = title
            if len(blocks) > 1:
                block_title += (f"  (periods {self.period[block[0]]}"
                                 f"-{self.period[block[-1]]})")

            print("=" * total_width)
            print(block_title)
            print("=" * total_width)
            print(header_line)
            print("-" * total_width)

            for name, values in variables:
                row_line = f"{name:<{label_width}}" + "".join(
                    f"{values[i]:>{col_width}.2f}" for i in block)
                print(row_line)

            print("=" * total_width)
            print()

    def print_current_state(self):
        self._print_table([len(self.period) - 1], "CURRENT STATE \u2014 Model LP")

    def print_history(self):
        self._print_table(range(len(self.period)), "SIMULATION RESULTS \u2014 Model LP")

    def check_consistency(self, tol=1e-6):
        #Household money holdings must equal money supplied, and long bonds
        #held must equal long bonds supplied, in every period.
        title = "STOCK-FLOW CONSISTENCY CHECK (H_h=H_s and BL_h=BL_s)"
        rows = []
        all_ok = True
        for i in range(len(self.period)):
            money_gap = self.H_h[i] - self.H_s[i]
            bond_gap = self.BL_h[i] - self.BL_s[i]
            ok = max(abs(money_gap), abs(bond_gap)) < tol
            all_ok = all_ok and ok
            symbol = "\u2713" if ok else "\u2717"
            rows.append(f"Period {self.period[i]:<3} "
                        f"H_h-H_s = {money_gap:>10.2e}   BL_h-BL_s = {bond_gap:>10.2e}   "
                        f"[{symbol}]")
        summary = ("All periods consistent." if all_ok
                   else "Inconsistency detected -- check the equations.")

        inner = max([len(title), len(summary)] + [len(r) for r in rows]) + 2
        top = "\u256d" + "\u2500" * inner + "\u256e"
        mid = "\u251c" + "\u2500" * inner + "\u2524"
        bottom = "\u2570" + "\u2500" * inner + "\u256f"

        print()
        print(top)
        print("\u2502" + title.center(inner) + "\u2502")
        print(mid)
        for row in rows:
            print("\u2502 " + row.ljust(inner - 1) + "\u2502")
        print(mid)
        print("\u2502" + summary.center(inner) + "\u2502")
        print(bottom)


def run_economy(G, alpha1, alpha2, theta, r_b, pBL, YD_0, V_0, B_h_0, BL_h_0,
                lambda_h, lambda_b, lambda_bl):
    #G, alpha1, alpha2, theta, r_b, and pBL are equal-length lists, one value per period.
    series = [G, alpha1, alpha2, theta, r_b, pBL]
    if len({len(x) for x in series}) != 1:
        raise ValueError("All time-series inputs must have the same length.")
    econ = LiquidityPreferenceEconomy(YD_0, V_0, B_h_0, BL_h_0, pBL[0], r_b[0])
    for values in zip(G, alpha1, alpha2, theta, r_b, pBL):
        econ.run_period(*values, lambda_h, lambda_b, lambda_bl)
    econ.print_history()
    econ.check_consistency()
    return econ


#Base LP illustration: pBL is fixed at 20, so capital gains are zero.
#The three constant portfolio weights sum to one.
econ = run_economy(
    G=[20, 20, 20], alpha1=[0.60, 0.60, 0.60], alpha2=[0.10, 0.10, 0.10],
    theta=[0.20, 0.20, 0.20], r_b=[0.03, 0.03, 0.03], pBL=[20, 20, 20],
    YD_0=80, V_0=200, B_h_0=70, BL_h_0=5,
    lambda_h=(0.15, 0, 0, 0),
    lambda_b=(0.35, 0, 0, 0),
    lambda_bl=(0.50, 0, 0, 0),
)

SIMULATION RESULTS — Model LP
Variable      Period 0    Period 1    Period 2    Period 3
----------------------------------------------------------
G                 0.00       20.00       20.00       20.00
Y                 0.00       88.00       86.46       86.79
T                 0.00       19.02       18.80       18.90
YD               80.00       76.08       75.19       75.62
YD^e             80.00       80.00       76.08       75.19
C                 0.00       68.00       66.46       66.79
CG                0.00        0.00        0.00        0.00
V               200.00      208.08      216.81      225.63
V^e             200.00      212.00      217.70      225.20
H^d              30.00       31.80       32.66       33.78
B^d              70.00       74.20       76.20       78.82
BL^d              5.00        5.30        5.44        5.63
H_h              30.00       33.88       40.61       46.81
B_h              70.00       74.20       76.20       78.82
BL_h              5.00    